In [7]:
import os
import pandas as pd

os.chdir('/home/jovyan/work/Documents/CASA/05_Data_Science_SS/_AssessmentFinal')
print(os.getcwd())  # confirma que dice /home/jovyan/work

/home/jovyan/work/Documents/CASA/05_Data_Science_SS/_AssessmentFinal


In [ ]:
#DFs ready to merge: 
# job_acc, 
# df_imd, 
# df_bornoutsideuk
# df_region  ---- region_name
# gdf ---- dist_to_london_km  dist_to_city_km
# df_busstops[['geo_code', 'bus_stops_per_km2']]

In [9]:
#----------JOB ACCESIBILITY PCT----------

job_acc_0 = pd.read_csv('data/access_employment_pt.csv')

# Filtrar Inglaterra y quedarte solo con las columnas relevantes
job_acc = job_acc_0[job_acc_0['geo_code'].str.startswith('E01')][
    ['geo_code', 'employment_pct_45', 'employment_pct_60']
].reset_index(drop=True)

print(job_acc.shape)
print(job_acc.describe())

(32844, 3)
       employment_pct_45  employment_pct_60
count       32844.000000       32844.000000
mean            0.705436           1.567662
std             1.833889           3.147786
min             0.000249           0.000249
25%             0.059532           0.145853
50%             0.169133           0.384978
75%             0.403573           1.018085
max            12.540768          16.700127


In [10]:
#----------IMD SCORES----------
df_imd = pd.read_excel('data/File_5_-_IoD2019_Scores.xlsx', sheet_name=1)

df_imd = df_imd[[
    'LSOA code (2011)',
    'Education, Skills and Training Score',
    'Health Deprivation and Disability Score',
    'Crime Score',
    'Barriers to Housing and Services Score'
]].rename(columns={
    'LSOA code (2011)': 'geo_code',
    'Education, Skills and Training Score': 'imd_education',
    'Health Deprivation and Disability Score': 'imd_health',
    'Crime Score': 'imd_crime',
    'Barriers to Housing and Services Score': 'imd_housing_barriers'
})

print(df_imd.shape)
print(df_imd.head(3))

(32844, 5)
    geo_code  imd_education  imd_health  imd_crime  imd_housing_barriers
0  E01000001          0.024      -1.654     -2.012                29.472
1  E01000002          0.063      -1.115     -2.343                24.412
2  E01000003          5.804      -0.102     -1.032                40.103


In [12]:
import pandas as pd
import os

def read_nomis(filepath):
    """Lee un CSV de Nomis, salta metadata, renombra mnemonic a geo_code"""
    df = pd.read_csv(filepath, skiprows=7)  # headers en fila 8 (index 7)
    df = df.rename(columns={df.columns[1]: 'geo_code'})
    df = df.dropna(subset=['geo_code'])  # elimina fila 9 vacía y filas al final
    df = df[df['geo_code'].str.startswith('E01')]
    return df.reset_index(drop=True)

In [13]:
def read_nomis2(filepath):
    """Lee un CSV de Nomis, salta metadata, renombra mnemonic a geo_code"""
    df = pd.read_csv(filepath, skiprows=5)  # headers en fila 8 (index 7)
    df = df.rename(columns={df.columns[1]: 'geo_code'})
    df = df.dropna(subset=['geo_code'])  # elimina fila 9 vacía y filas al final
    df = df[df['geo_code'].str.startswith('E01')]
    return df.reset_index(drop=True)

In [14]:
df_bornoutsideuk = read_nomis('data/census_pct_bornoutsideuk.csv')

print(df_bornoutsideuk.shape)
print(df_bornoutsideuk.columns.tolist())
print(df_bornoutsideuk.head(3))

(33755, 14)
['2021 super output area - lower layer', 'geo_code', 'Total: All usual residents', 'Europe: EU countries', 'Europe: EU countries: European Union EU14', 'Europe: EU countries: European Union EU8', 'Europe: EU countries: European Union EU2', 'Europe: EU countries: All other EU countries', 'Europe: Non-EU countries - All other non-EU countries', 'Africa', 'Middle East and Asia', 'The Americas and the Caribbean', 'Antarctica and Oceania (including Australasia) and Other', 'British Overseas']
  2021 super output area - lower layer   geo_code  Total: All usual residents  \
0                      Hartlepool 001A  E01011954                       100.0   
1                      Hartlepool 001B  E01011969                       100.0   
2                      Hartlepool 001C  E01011970                       100.0   

   Europe: EU countries  Europe: EU countries: European Union EU14  \
0                   0.9                                        0.4   
1                   0.7       

In [ ]:
print(df_bornoutsideuk['geo_code'].str[:3].value_counts())

geo_code
E01    33755
Name: count, dtype: int64


In [ ]:
df_bornoutsideuk['pct_born_outside_uk'] = (
    df_bornoutsideuk['Europe: EU countries'] +
    df_bornoutsideuk['Europe: Non-EU countries - All other non-EU countries'] +
    df_bornoutsideuk['Africa'] +
    df_bornoutsideuk['Middle East and Asia'] +
    df_bornoutsideuk['The Americas and the Caribbean'] +
    df_bornoutsideuk['Antarctica and Oceania (including Australasia) and Other'] +
    df_bornoutsideuk['British Overseas']
)

df_bornoutsideuk = df_bornoutsideuk[['geo_code', 'pct_born_outside_uk']]
print(df_bornoutsideuk.describe())

       pct_born_outside_uk
count         33755.000000
mean             16.591711
std              14.973803
min               0.600000
25%               5.400000
50%              10.500000
75%              23.450000
max              89.500000


In [ ]:
df_density= read_nomis('data/census_density.csv')

print(df_density.shape)
print(df_density.columns.tolist())
print(df_density.head(3))

(33755, 3)
['2021 super output area - lower layer', 'geo_code', 'Usual residents per square kilometre']
  2021 super output area - lower layer   geo_code  \
0                      Hartlepool 001A  E01011954   
1                      Hartlepool 001B  E01011969   
2                      Hartlepool 001C  E01011970   

   Usual residents per square kilometre  
0                                5432.9  
1                                1430.7  
2                                2913.9  


In [ ]:
df_density = df_density.rename(columns={'Usual residents per square kilometre': 'density'})
print(df_density)

        geo_code  density
0      E01011954   5432.9
1      E01011969   1430.7
2      E01011970   2913.9
3      E01011971   6049.4
4      E01033465   4311.9
...          ...      ...
33750  E01029342     39.7
33751  E01029343    203.6
33752  E01029325     32.6
33753  E01029328     41.1
33754  E01029338      7.2

[33755 rows x 2 columns]


In [ ]:
df_age= read_nomis2('data/census_pct_age152560mas.csv')

print(df_age.shape)
print(df_age.columns.tolist())
print(df_age.head(3))

(33755, 10)
['2021 super output area - lower layer', 'geo_code', 'Total', 'Aged 15 to 19 years', 'Aged 20 to 24 years', 'Aged 65 to 69 years', 'Aged 70 to 74 years', 'Aged 75 to 79 years', 'Aged 80 to 84 years', 'Aged 85 years and over']
  2021 super output area - lower layer   geo_code  Total  Aged 15 to 19 years  \
0                      Hartlepool 001A  E01011954  100.0                  5.6   
1                      Hartlepool 001B  E01011969  100.0                  4.9   
2                      Hartlepool 001C  E01011970  100.0                  4.1   

   Aged 20 to 24 years  Aged 65 to 69 years  Aged 70 to 74 years  \
0                  6.3                  4.2                  4.0   
1                  4.7                  7.9                  8.5   
2                  5.0                  9.9                  9.7   

   Aged 75 to 79 years  Aged 80 to 84 years  Aged 85 years and over  
0                  2.0                  2.2                     1.8  
1                  6.1  

In [ ]:
# % aged 16-24 (usamos 15-19 + 20-24 como proxy)
df_age['pct_aged_16_24'] = (
    df_age['Aged 15 to 19 years'] + 
    df_age['Aged 20 to 24 years']
)

# % aged 65+
df_age['pct_aged_65plus'] = (
    df_age['Aged 65 to 69 years'] +
    df_age['Aged 70 to 74 years'] +
    df_age['Aged 75 to 79 years'] +
    df_age['Aged 80 to 84 years'] +
    df_age['Aged 85 years and over']
)

df_age = df_age[['geo_code', 'pct_aged_16_24', 'pct_aged_65plus']]
print(df_age.head(3))

    geo_code  pct_aged_16_24  pct_aged_65plus
0  E01011954            11.9             14.2
1  E01011969             9.6             31.2
2  E01011970             9.1             29.2


In [ ]:
df_degree= read_nomis('data/census_pct_degrelevel_nocualif.csv')

print(df_degree.shape)
print(df_degree.columns.tolist())
print(df_degree.head(3))

 

(33755, 5)
['2021 super output area - lower layer', 'geo_code', 'Total: All usual residents aged 16 years and over', 'No qualifications', 'Level 4 qualifications or above']
  2021 super output area - lower layer   geo_code  \
0                      Hartlepool 001A  E01011954   
1                      Hartlepool 001B  E01011969   
2                      Hartlepool 001C  E01011970   

   Total: All usual residents aged 16 years and over  No qualifications  \
0                                              100.0               24.1   
1                                              100.0               21.1   
2                                              100.0               16.8   

   Level 4 qualifications or above  
0                             18.8  
1                             22.8  
2                             30.6  


In [ ]:
df_degree = df_degree.rename(columns={'No qualifications': 'pct_noqualif'})
df_degree = df_degree.rename(columns={'Level 4 qualifications or above': 'pct_level4andabove'})
df_degree = df_degree[['geo_code', 'pct_noqualif', 'pct_level4andabove']]
print(df_degree)

        geo_code  pct_noqualif  pct_level4andabove
0      E01011954          24.1                18.8
1      E01011969          21.1                22.8
2      E01011970          16.8                30.6
3      E01011971          10.3                33.0
4      E01033465          10.9                34.6
...          ...           ...                 ...
33750  E01029342          13.9                35.1
33751  E01029343          21.5                25.6
33752  E01029325          17.3                30.5
33753  E01029328          18.9                31.5
33754  E01029338          18.0                32.8

[33755 rows x 3 columns]


In [ ]:
df_ethnicity= read_nomis('data/census_pct_ethnicminority.csv')

print(df_ethnicity.shape)
print(df_ethnicity.columns.tolist())
print(df_ethnicity.head(3))


(33755, 11)
['2021 super output area - lower layer', 'geo_code', 'Total: All usual residents', 'Asian, Asian British or Asian Welsh', 'Black, Black British, Black Welsh, Caribbean or African', 'Mixed or Multiple ethnic groups', 'White: Irish', 'White: Gypsy or Irish Traveller', 'White: Roma', 'White: Other White', 'Other ethnic group']
  2021 super output area - lower layer   geo_code  Total: All usual residents  \
0                      Hartlepool 001A  E01011954                       100.0   
1                      Hartlepool 001B  E01011969                       100.0   
2                      Hartlepool 001C  E01011970                       100.0   

   Asian, Asian British or Asian Welsh  \
0                                  0.8   
1                                  0.4   
2                                  0.7   

   Black, Black British, Black Welsh, Caribbean or African  \
0                                                0.0         
1                                           

In [ ]:
df_ethnicity = df_ethnicity[['geo_code', 'pct_ethnicityminority']]
print(df_ethnicity.head(15))





     geo_code  pct_ethnicityminority
0   E01011954                    2.4
1   E01011969                    1.6
2   E01011970                    1.7
3   E01011971                    1.4
4   E01033465                    3.0
5   E01033467                    3.3
6   E01011952                    4.6
7   E01011953                    3.1
8   E01011991                    2.9
9   E01011992                    8.2
10  E01011993                    2.3
11  E01011994                    5.8
12  E01011955                    9.0
13  E01011957                    6.9
14  E01032540                    3.8


In [ ]:
df_loneparenthousehold= read_nomis('data/census_pct_loneparenthousehold.csv')
df_loneparenthousehold = df_loneparenthousehold.rename(columns={'Single family household: Lone parent family': 'pct_loneparenthousehold'})

df_loneparenthousehold = df_loneparenthousehold[['geo_code', 'pct_loneparenthousehold']]

print(df_loneparenthousehold.shape)
print(df_loneparenthousehold.columns.tolist())
print(df_loneparenthousehold.head(3))



(33755, 2)
['geo_code', 'pct_loneparenthousehold']
    geo_code  pct_loneparenthousehold
0  E01011954                     19.5
1  E01011969                      9.3
2  E01011970                      8.0


In [ ]:


df_nocar= read_nomis('data/census_pct_nocaravailability.csv')
df_nocar = df_nocar.rename(columns={'No cars or vans in household': 'pct_nocar'})

df_nocar = df_nocar[['geo_code', 'pct_nocar']]

print(df_nocar.shape)
print(df_nocar.columns.tolist())

print(df_nocar.head(3))


(33755, 2)
['geo_code', 'pct_nocar']
    geo_code  pct_nocar
0  E01011954       31.7
1  E01011969       14.4
2  E01011970        9.9


In [ ]:


df_overcrowded= read_nomis('data/census_pct_overcrowded.csv')

df_overcrowded['pct_overcrowded'] = (
    df_overcrowded['Occupancy rating of bedrooms: -1'] + 
    df_overcrowded['Occupancy rating of bedrooms: -2 or less']
)


df_overcrowded = df_overcrowded[['geo_code', 'pct_overcrowded']]

print(df_overcrowded.shape)
print(df_overcrowded.columns.tolist())

print(df_overcrowded.head(3))


(33755, 2)
['geo_code', 'pct_overcrowded']
    geo_code  pct_overcrowded
0  E01011954              2.7
1  E01011969              0.5
2  E01011970              0.8


In [ ]:

df_rent= read_nomis('data/census_pct_owned_socialrent_privaterent_.csv')

df_rent = df_rent.rename(columns={'Social rented': 'pct_social_rented'})
df_rent = df_rent.rename(columns={'Private rented': 'pct_private_rented'})

df_rent = df_rent[['geo_code', 'pct_social_rented', 'pct_private_rented']]



print(df_rent.head(3))

    geo_code  pct_social_rented  pct_private_rented
0  E01011954               27.2                17.0
1  E01011969                3.7                11.8
2  E01011970                1.2                 8.2


In [ ]:


df_parttime= read_nomis('data/census_pct_parttime.csv')


df_parttime = df_parttime.rename(columns={'Part-time': 'pct_part_time'})


df_parttime = df_parttime[['geo_code', 'pct_part_time']]


print(df_parttime.head(3))

    geo_code  pct_part_time
0  E01011954           34.8
1  E01011969           26.3
2  E01011970           27.6


In [ ]:


df_selfemployed= read_nomis('data/census_pct_selfemployed.csv')


df_selfemployed['pct_selfemployed'] = (
    df_selfemployed['Economically active (excluding full-time students):In employment:Self-employed with employees'] + 
    df_selfemployed['Economically active (excluding full-time students):In employment:Self-employed without employees']
)

df_selfemployed = df_selfemployed[['geo_code', 'pct_selfemployed']]


print(df_selfemployed.head(3))

    geo_code  pct_selfemployed
0  E01011954               4.4
1  E01011969               4.9
2  E01011970               5.9


In [ ]:
import pandas as pd

# Lista de todos los dfs y sus columnas
dfs = [
    (df_density,            ['geo_code', 'density']),
    (df_age,                ['geo_code', 'pct_aged_16_24', 'pct_aged_65plus']),
    (df_bornoutsideuk,      ['geo_code', 'pct_born_outside_uk']),
    (df_degree,             ['geo_code', 'pct_noqualif', 'pct_level4andabove']),
    (df_ethnicity,          ['geo_code', 'pct_ethnicityminority']),
    (df_loneparenthousehold,['geo_code', 'pct_loneparenthousehold']),
    (df_nocar,              ['geo_code', 'pct_nocar']),
    (df_overcrowded,        ['geo_code', 'pct_overcrowded']),
    (df_rent,               ['geo_code', 'pct_social_rented', 'pct_private_rented']),
    (df_parttime,           ['geo_code', 'pct_part_time']),
    (df_selfemployed,       ['geo_code', 'pct_selfemployed']),
]

# Merge progresivo
df_census = dfs[0][0][dfs[0][1]].copy()

for df, cols in dfs[1:]:
    df_census = df_census.merge(df[cols], on='geo_code', how='left')

print(df_census.shape)
print(df_census.columns.tolist())
print(df_census.isnull().sum())

(33755, 15)
['geo_code', 'density', 'pct_aged_16_24', 'pct_aged_65plus', 'pct_born_outside_uk', 'pct_noqualif', 'pct_level4andabove', 'pct_ethnicityminority', 'pct_loneparenthousehold', 'pct_nocar', 'pct_overcrowded', 'pct_social_rented', 'pct_private_rented', 'pct_part_time', 'pct_selfemployed']
geo_code                   0
density                    0
pct_aged_16_24             0
pct_aged_65plus            0
pct_born_outside_uk        0
pct_noqualif               0
pct_level4andabove         0
pct_ethnicityminority      0
pct_loneparenthousehold    0
pct_nocar                  0
pct_overcrowded            0
pct_social_rented          0
pct_private_rented         0
pct_part_time              0
pct_selfemployed           0
dtype: int64


In [ ]:
df_census.to_csv('data/census_merged.csv', index=False)
print("Guardado!")

Guardado!


In [16]:
df_ruc = pd.read_excel('data/260331rucallsupplementarytables.xlsx', sheet_name='Table 3B', skiprows=2)

df_ruc = df_ruc[['LSOA21CD', 'RUC21CD detailed', 'RUC21NM detailed']].rename(columns={
    'LSOA21CD': 'geo_code',
    'RUC21CD detailed': 'ruc_code',
    'RUC21NM detailed': 'ruc_name'
})

df_ruc = df_ruc[df_ruc['geo_code'].str.startswith('E01')]

print(df_ruc.shape)
print(df_ruc['ruc_code'].value_counts())

(33755, 3)
ruc_code
UN1     26198
RLN1     2038
RSN1     1690
UF2      1618
RSF2      536
RLF2      495
RSF3      384
UF3       383
RLF3      303
RSF4       59
RLF4       51
Name: count, dtype: int64


In [17]:
df_ruc = df_ruc[['geo_code', 'ruc_code']]
print(df_ruc.head(3))

    geo_code ruc_code
0  E01000001      UN1
1  E01000002      UN1
2  E01000003      UN1


In [18]:
# Ver el IMD original completo antes de que lo filtraste
df_imd_full = pd.read_excel('data/File_5_-_IoD2019_Scores.xlsx', sheet_name=1)
print(df_imd_full[['LSOA code (2011)', 'Local Authority District code (2019)']].head(3))

  LSOA code (2011) Local Authority District code (2019)
0        E01000001                            E09000001
1        E01000002                            E09000001
2        E01000003                            E09000001


In [20]:
df_region = pd.read_csv('data/LSOA_(2021)_to_Built_Up_Area_to_Local_Authority_District_to_Region_(December_2022)_Lookup_in_England_and_Wales_v2.csv')

df_region = df_region[['LSOA21CD', 'RGN22CD', 'RGN22NM']].rename(columns={
    'LSOA21CD': 'geo_code',
    'RGN22CD': 'region_code',
    'RGN22NM': 'region_name'
})

df_region = df_region[df_region['geo_code'].str.startswith('E01')]

print(df_region.shape)
print(df_region['region_name'].value_counts())

(33755, 3)
region_name
South East                  5571
London                      4994
North West                  4567
East of England             3758
West Midlands               3574
South West                  3407
Yorkshire and The Humber    3355
East Midlands               2847
North East                  1682
Name: count, dtype: int64


In [28]:
import geopandas as gpd

# Recargar limpio
gdf = gpd.read_file('data/Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BSC_V4_6894679968818356315.geojson')

# Filtrar Inglaterra primero ANTES de reprojectar
gdf = gdf[gdf['LSOA21CD'].str.startswith('E01')].copy()
gdf = gdf.rename(columns={'LSOA21CD': 'geo_code'})

# Verificar geometría
print(f"Total: {len(gdf)}")
print(f"Geometrías válidas: {gdf.geometry.notna().sum()}")
print(f"CRS: {gdf.crs}")
print(gdf.geometry.head(3))



Total: 33755
Geometrías válidas: 33755
CRS: EPSG:4326
0    POLYGON ((-0.09474 51.5206, -0.09546 51.51544,...
1    POLYGON ((-0.0881 51.51941, -0.09546 51.51544,...
2    POLYGON ((-0.09453 51.52205, -0.09274 51.52139...
Name: geometry, dtype: geometry


In [32]:
print(gdf.head(3))

   FID   geo_code             LSOA21NM LSOA21NMW   BNG_E   BNG_N       LAT  \
0    1  E01000001  City of London 001A            532123  181632  51.51817   
1    2  E01000002  City of London 001B            532480  181715  51.51883   
2    3  E01000003  City of London 001C            532239  182033  51.52174   

      LONG                              GlobalID  \
0 -0.09715  3478c558-3297-4e2b-979e-e29dd9ff3bf5   
1 -0.09197  f2072109-b1ae-426c-b166-083cc32f1789   
2 -0.09533  a9009c33-9b6b-4230-ba62-fc3264806de4   

                                            geometry  
0  POLYGON ((-0.09474 51.5206, -0.09546 51.51544,...  
1  POLYGON ((-0.0881 51.51941, -0.09546 51.51544,...  
2  POLYGON ((-0.09453 51.52205, -0.09274 51.52139...  


In [33]:
# Filtrar solo Inglaterra y reprojectar a metros
gdf = gdf[gdf['geo_code'].str.startswith('E01')].copy()
gdf = gdf.to_crs(epsg=27700)

# Calcular distancias
from shapely.geometry import Point

london = Point(530000, 180000)
gdf['dist_to_london_km'] = gdf.geometry.centroid.distance(london) / 1000

city_points = [
    Point(530000, 180000),  # London
    Point(406900, 286800),  # Birmingham
    Point(383500, 398000),  # Manchester
    Point(430500, 433500),  # Leeds
    Point(435300, 387200),  # Sheffield
    Point(336700, 390200),  # Liverpool
    Point(358500, 173000),  # Bristol
    Point(424600, 564200),  # Newcastle
    Point(457100, 340100),  # Nottingham
    Point(459500, 304900),  # Leicester
]

gdf['dist_to_city_km'] = gdf.geometry.centroid.apply(
    lambda c: min(c.distance(p) for p in city_points) / 1000
)

print(gdf[['geo_code', 'dist_to_london_km', 'dist_to_city_km']].describe())

       dist_to_london_km  dist_to_city_km
count       33755.000000     33755.000000
mean          158.777169        41.991580
std           112.132107        38.998454
min             0.484771         0.182654
25%            53.814998        12.958484
50%           155.908150        28.674076
75%           256.067463        60.167313
max           492.666795       312.175851


In [35]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# Cargar NaPTAN
naptan = pd.read_csv('data/Stops.csv', low_memory=False)
print(naptan.columns.tolist())
print(naptan.shape)
print(naptan[['Easting', 'Northing', 'Status']].head())

['ATCOCode', 'NaptanCode', 'PlateCode', 'CleardownCode', 'CommonName', 'CommonNameLang', 'ShortCommonName', 'ShortCommonNameLang', 'Landmark', 'LandmarkLang', 'Street', 'StreetLang', 'Crossing', 'CrossingLang', 'Indicator', 'IndicatorLang', 'Bearing', 'NptgLocalityCode', 'LocalityName', 'ParentLocalityName', 'GrandParentLocalityName', 'Town', 'TownLang', 'Suburb', 'SuburbLang', 'LocalityCentre', 'GridType', 'Easting', 'Northing', 'Longitude', 'Latitude', 'StopType', 'BusStopType', 'TimingStatus', 'DefaultWaitTime', 'Notes', 'NotesLang', 'AdministrativeAreaCode', 'CreationDateTime', 'ModificationDateTime', 'RevisionNumber', 'Modification', 'Status']
(434581, 43)
   Easting  Northing  Status
0   359403    172513  active
1   359452    172461  active
2   359581    172304  active
3   359438    172366  active
4   359609    172383  active


In [36]:
# Filtrar solo paradas activas de autobús en Inglaterra
naptan_bus = naptan[
    (naptan['Status'] == 'active') &
    (naptan['StopType'] == 'BCT') &  # Bus/Coach Tram stops only
    (naptan['ATCOCode'].str[:3].str.isdigit())  # England only (numeric area codes)
].copy()

print(f"Bus stops after filter: {len(naptan_bus)}")

# Convertir a GeoDataFrame (ya en BNG metros)
gdf_stops = gpd.GeoDataFrame(
    naptan_bus,
    geometry=gpd.points_from_xy(naptan_bus['Easting'], naptan_bus['Northing']),
    crs='EPSG:27700'
)

# Spatial join — cada parada al LSOA que la contiene
stops_joined = gpd.sjoin(gdf_stops[['geometry']], gdf[['geo_code', 'geometry']], 
                          how='left', predicate='within')

# Contar paradas por LSOA
stops_count = stops_joined.groupby('geo_code').size().reset_index(name='bus_stops_count')

# Calcular área en km² y densidad
gdf['area_km2'] = gdf.geometry.area / 1_000_000
df_busstops = gdf[['geo_code', 'area_km2']].merge(stops_count, on='geo_code', how='left')
df_busstops['bus_stops_count'] = df_busstops['bus_stops_count'].fillna(0)
df_busstops['bus_stops_per_km2'] = df_busstops['bus_stops_count'] / df_busstops['area_km2']

print(df_busstops[['bus_stops_per_km2']].describe())
print(f"LSOAs with 0 stops: {(df_busstops['bus_stops_count'] == 0).sum()}")

Bus stops after filter: 371135
       bus_stops_per_km2
count       33755.000000
mean           14.540343
std            13.254449
min             0.000000
25%             3.980623
50%            11.736127
75%            21.497553
max           277.644539
LSOAs with 0 stops: 1621


In [37]:
# Guardar df_busstops
df_busstops[['geo_code', 'bus_stops_per_km2']].to_csv('data/bus_stops_per_km2.csv', index=False)
print("✅ bus stops saved")

✅ bus stops saved


In [38]:
df_unemployment = read_nomis('data/census_value_unemployment.csv')

print(df_unemployment.shape)
print(df_unemployment.columns.tolist())
print(df_unemployment.head(3))

(33755, 5)
['2021 super output area - lower layer', 'geo_code', 'Total: All usual residents aged 16 years and over', 'Economically active (excluding full-time students)', 'Economically active (excluding full-time students): Unemployed']
  2021 super output area - lower layer   geo_code  \
0                      Hartlepool 001A  E01011954   
1                      Hartlepool 001B  E01011969   
2                      Hartlepool 001C  E01011970   

   Total: All usual residents aged 16 years and over  \
0                                             1758.0   
1                                             1151.0   
2                                              935.0   

   Economically active (excluding full-time students)  \
0                                              984.0    
1                                              580.0    
2                                              527.0    

   Economically active (excluding full-time students): Unemployed  
0                           

In [40]:

df_unemployment = df_unemployment.rename(columns={
    df_unemployment.columns[1]: 'geo_code',
    'Economically active (excluding full-time students)': 'econ_active',
    'Economically active (excluding full-time students): Unemployed': 'unemployed'
})
df_unemployment = df_unemployment[df_unemployment['geo_code'].str.startswith('E01')].copy()

# Calcular Y variable
df_unemployment['pct_unemployed'] = (df_unemployment['unemployed'] / df_unemployment['econ_active']) * 100

# Verificar
print(df_unemployment[['geo_code', 'pct_unemployed']].describe())
print(f"NaN en pct_unemployed: {df_unemployment['pct_unemployed'].isna().sum()}")

# Guardar solo lo necesario
df_unemployment = df_unemployment[['geo_code', 'pct_unemployed']].copy()
df_unemployment.to_csv('data/unemployment.csv', index=False)
print("✅ unemployment saved")

       pct_unemployed
count    33755.000000
mean         4.966262
std          2.752933
min          0.000000
25%          3.041684
50%          4.236200
75%          6.224695
max         27.153558
NaN en pct_unemployed: 0
✅ unemployment saved


In [42]:
df_census = pd.read_csv('data/census_merged.csv')
print(df_census.shape)
print(df_census.columns.tolist())

(33755, 15)
['geo_code', 'density', 'pct_aged_16_24', 'pct_aged_65plus', 'pct_born_outside_uk', 'pct_noqualif', 'pct_level4andabove', 'pct_ethnicityminority', 'pct_loneparenthousehold', 'pct_nocar', 'pct_overcrowded', 'pct_social_rented', 'pct_private_rented', 'pct_part_time', 'pct_selfemployed']


In [43]:
# Master merge — todo desde kernel
dfs_to_merge = [
    df_imd[['geo_code', 'imd_education', 'imd_health', 'imd_crime', 'imd_housing_barriers']],
    job_acc[['geo_code', 'employment_pct_45', 'employment_pct_60']],
    df_region[['geo_code', 'region_name']],
    df_ruc[['geo_code', 'ruc_code']],
    gdf[['geo_code', 'dist_to_london_km', 'dist_to_city_km']],
    df_busstops[['geo_code', 'bus_stops_per_km2']],
    df_unemployment[['geo_code', 'pct_unemployed']],
]

df_master = df_census.copy()
for df in dfs_to_merge:
    df_master = df_master.merge(df, on='geo_code', how='left')

print(df_master.shape)
print(df_master.isnull().sum())

(33755, 27)
geo_code                      0
density                       0
pct_aged_16_24                0
pct_aged_65plus               0
pct_born_outside_uk           0
pct_noqualif                  0
pct_level4andabove            0
pct_ethnicityminority         0
pct_loneparenthousehold       0
pct_nocar                     0
pct_overcrowded               0
pct_social_rented             0
pct_private_rented            0
pct_part_time                 0
pct_selfemployed              0
imd_education              1945
imd_health                 1945
imd_crime                  1945
imd_housing_barriers       1945
employment_pct_45          1945
employment_pct_60          1945
region_name                   0
ruc_code                      0
dist_to_london_km             0
dist_to_city_km               0
bus_stops_per_km2             0
pct_unemployed                0
dtype: int64


In [44]:
df_master = df_master.dropna(subset=['employment_pct_45', 'imd_education'])
print(df_master.shape)

(31810, 27)


In [46]:
df_inactive = pd.read_csv('data/census_pct_inactive.csv', skiprows=7)
print(df_inactive.columns.tolist())
print(df_inactive.shape)

['2021 super output area - lower layer', 'mnemonic', 'Total: All usual residents aged 16 years and over', 'Economically inactive']
(35677, 4)


In [47]:
def read_nomis(filepath):
    df = pd.read_csv(filepath, skiprows=7)
    df = df.rename(columns={df.columns[1]: 'geo_code'})
    df = df.dropna(subset=['geo_code'])
    df = df[df['geo_code'].str.startswith('E01')]
    return df.reset_index(drop=True)

df_inactive = read_nomis('data/census_pct_inactive.csv')
df_inactive = df_inactive.rename(columns={'Economically inactive': 'pct_inactive'})
df_inactive = df_inactive[['geo_code', 'pct_inactive']]

print(df_inactive.shape)
print(df_inactive['pct_inactive'].describe())

(33755, 2)
count    33755.000000
mean        39.110313
std          8.116201
min          4.900000
25%         34.300000
50%         39.300000
75%         44.100000
max         83.000000
Name: pct_inactive, dtype: float64


In [48]:
df_master = df_master.merge(df_inactive, on='geo_code', how='left')
print(df_master.shape)
print(df_master['pct_inactive'].isnull().sum())

# Guardar
df_master.to_csv('data/master_dataset.csv', index=False)
print("✅ master_dataset.csv updated")

(31810, 28)
0
✅ master_dataset.csv updated
